In [22]:
!pip install transformers datasets accelerate

In [23]:
import pandas as pd
import torch

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [24]:
data = {
    "text": [

        # Real Profiles
        "Computer science student interested in AI and cybersecurity",
        "Software engineer passionate about backend development",
        "Data scientist working on machine learning applications",
        "Student at IIT Delhi interested in robotics",
        "Frontend developer and UI designer",
        "Research assistant in artificial intelligence",
        "Machine learning enthusiast and open source contributor",
        "Cybersecurity analyst and ethical hacker",
        "Mobile app developer using Flutter",
        "Graduate student researching NLP systems",

        # Suspisious Profiles
        "Crypto millionaire. DM now for passive income",
        "Earn money fast with this secret AI strategy",
        "NFT investor | millionaire mindset | success coach",
        "Click link below to become financially free",
        "AI influencer helping people get rich instantly",
        "DM now for guaranteed crypto profits",
        "Trading expert with secret investment formula",
        "Make $10000 every week without effort",
        "Passive income king and crypto mastermind",
        "Unlock hidden wealth using AI automation"
    ],

    "label": [
        0,0,0,0,0,0,0,0,0,0,
        1,1,1,1,1,1,1,1,1,1
    ]
}

In [25]:
df = pd.DataFrame(data)

print(df.head())
print()
print("Dataset Size:", len(df))

                                                text  label
0  Computer science student interested in AI and ...      0
1  Software engineer passionate about backend dev...      0
2  Data scientist working on machine learning app...      0
3        Student at IIT Delhi interested in robotics      0
4                 Frontend developer and UI designer      0

Dataset Size: 20


In [26]:
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)
print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [27]:
dataset = Dataset.from_pandas(df)
print(dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 20
})


In [28]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=64
    )

In [29]:
tokenized_dataset = dataset.map(tokenize_function)
print(tokenized_dataset)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 20
})


In [30]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.2,seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 16
})
Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 4
})


In [31]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=2)
print("DistilBERT model loaded successfully.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded successfully.


In [32]:
training_args = TrainingArguments(output_dir="./results",num_train_epochs=3,per_device_train_batch_size=4,per_device_eval_batch_size=4,eval_strategy="epoch",save_strategy="epoch",logging_dir="./logs",logging_steps=1)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [33]:
trainer = Trainer(model=model,args=training_args,train_dataset=train_dataset,eval_dataset=test_dataset)
print("Trainer initialized successfully.")

Trainer initialized successfully.


In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.683599,0.658682
2,0.625028,0.607632
3,0.477384,0.574488


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12, training_loss=0.6129469126462936, metrics={'train_runtime': 24.7292, 'train_samples_per_second': 1.941, 'train_steps_per_second': 0.485, 'total_flos': 794804391936.0, 'train_loss': 0.6129469126462936, 'epoch': 3.0})

In [35]:
print(next(model.parameters()).device)

cuda:0


In [42]:
def predict_profile_text(text):
    model.eval()
    model_device = next(model.parameters()).device
    inputs = tokenizer(text,return_tensors="pt",truncation=True,padding=True,max_length=64)
    inputs = {
        k: v.to(model_device)
        for k, v in inputs.items()
    }
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits,dim=1)
        confidence, prediction = torch.max(probabilities,dim=1)
    label_map = {
        0: "real",
        1: "suspicious"
    }
    predicted_label = label_map[
        prediction.item()
    ]
    confidence_score = confidence.item() * 100
    return predicted_label, confidence_score

In [43]:
test_text = "Crypto millionaire 🚀 DM now for passive income"

prediction, confidence = predict_profile_text(test_text
                                              )
print("Prediction:", prediction)
print(f"Confidence: {confidence:.2f}%")

Prediction: suspicious
Confidence: 62.65%


In [44]:
test_text = "Computer science student interested in machine learning and cybersecurity"

prediction, confidence = predict_profile_text(test_text)
print("Prediction:", prediction)
print(f"Confidence: {confidence:.2f}%")

Prediction: real
Confidence: 60.57%


In [48]:
def multimodal_profile_analysis(image_prediction,image_confidence,text_prediction,text_confidence):
    image_score = (image_confidence / 100 if image_prediction == "fake" else 1 - (image_confidence / 100))
    text_score = (text_confidence / 100 if text_prediction == "suspicious" else 1 - (text_confidence / 100))
    final_score = (0.6 * image_score + 0.4 * text_score)

    if final_score >= 0.5:
        verdict = "Synthetic Profile Likely"
    else:
        verdict = "Profile Appears Genuine"

    return verdict, final_score

In [49]:
verdict, score = multimodal_profile_analysis(image_prediction="fake",image_confidence=100,text_prediction="suspicious",text_confidence=66.94)
print("Final Verdict:", verdict)
print(f"Risk Score: {score:.2f}")

Final Verdict: Synthetic Profile Likely
Risk Score: 0.87
